##### 11D — Leakage Validation

In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/train_features.csv",
    low_memory=False,
    dtype={"StateHoliday": "string"}
)

df = df.sort_values(["Store", "Date"])

print(
    df[
        [
            "Store",
            "Date",
            "Sales",
            "Sales_Lag_1",
            "Sales_Lag_7",
            "Sales_Lag_14",
            "Sales_Rolling_Mean_7",
            "Sales_Rolling_Mean_14",
            "Sales_Rolling_Mean_30"
        ]
    ].head(20)
)

    Store        Date  Sales  Sales_Lag_1  Sales_Lag_7  Sales_Lag_14  \
0       1  2013-01-01      0          NaN          NaN           NaN   
1       1  2013-01-02   5530          0.0          NaN           NaN   
2       1  2013-01-03   4327       5530.0          NaN           NaN   
3       1  2013-01-04   4486       4327.0          NaN           NaN   
4       1  2013-01-05   4997       4486.0          NaN           NaN   
5       1  2013-01-06      0       4997.0          NaN           NaN   
6       1  2013-01-07   7176          0.0          NaN           NaN   
7       1  2013-01-08   5580       7176.0          0.0           NaN   
8       1  2013-01-09   5471       5580.0       5530.0           NaN   
9       1  2013-01-10   4892       5471.0       4327.0           NaN   
10      1  2013-01-11   4881       4892.0       4486.0           NaN   
11      1  2013-01-12   4952       4881.0       4997.0           NaN   
12      1  2013-01-13      0       4952.0          0.0          

##### 11D.2 — Check missing values

In [2]:
lag_columns = [
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30"
]

print(df[lag_columns].isna().sum())

Sales_Lag_1               1115
Sales_Lag_7               7805
Sales_Lag_14             15610
Sales_Rolling_Mean_7      7805
Sales_Rolling_Mean_14    15610
Sales_Rolling_Mean_30    33450
dtype: int64


##### 11D.3 — Most important leakage test

In [3]:
correlation = df[
    ["Sales", "Sales_Lag_1", "Sales_Lag_7", "Sales_Lag_14"]
].corr()

print(correlation)

                 Sales  Sales_Lag_1  Sales_Lag_7  Sales_Lag_14
Sales         1.000000     0.278360     0.674894      0.793680
Sales_Lag_1   0.278360     1.000000     0.171927      0.258113
Sales_Lag_7   0.674894     0.171927     1.000000      0.674187
Sales_Lag_14  0.793680     0.258113     0.674187      1.000000


## 11D — Feature Leakage Validation

Lag and rolling features were validated to ensure that only historical
sales information is used.

The rolling calculations use `shift(1)` before calculating rolling means,
preventing the current day's Sales from entering its own features.

The resulting correlations with Sales were:

- Sales_Lag_1: 0.278
- Sales_Lag_7: 0.675
- Sales_Lag_14: 0.794

Missing values at the beginning of each store's history are expected because
sufficient historical observations are not available.

No evidence of target leakage was identified in the lag and rolling feature
construction.